# Dormant 3+ Year Reactivation Personalization — Power Analysis

**Experiment:** Personalization vs. No Personalization for Dormant 3+ Year Reactivation Campaigns (A/B) · **Owner:** Sergio Oyola  
**Primary metric:** Reactivation Rate (Fix OR Direct Buy) · **Randomization unit:** `client_id` · **Allocation point:** campaign send, restricted to the Dormant 3+ yrs population


## Eligibility & Population

The eligible population is clients currently in the **Dormant 3+ years** lifecycle bucket — no checkout in more than **1095 days (3 years)**. This sits on top of the standard lifecycle states already tracked in `curated.checkout_based_client_state_journal`:

| Bucket | Definition |
|---|---|
| Active | Last checkout ≤ 120 days ago |
| Lapsed | Last checkout 120–365 days ago |
| Dormant | Last checkout > 365 days ago |
| **Dormant 3+ yrs (eligible pool)** | Last checkout > 1095 days ago — a subset of Dormant |

`client_state_detail` on the journal table (values `Engaged` / `Lapsed` / `Dormant` / `Never Active`) already implements the Active/Lapsed/Dormant boundaries. The **1095-day cut has no pre-built bucket**, so it's derived by layering `days_since_last_checkout` (from `last_buyable_checkout_ts`) on top of `client_state_detail = 'Dormant'`.


## A/B Design Choices
- **2 arms** (Control = No Personalization / Treatment = Personalization), equal **50/50 split**.
- **Single planned comparison** (Treatment vs. Control); `alpha = 0.05`.
- **Two-sided** test: sizing for a +MDE gives symmetric power to detect harm of the same magnitude, so the Early Stop / harm threshold mirrors the committed MDE (same magnitude, opposite sign).

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)

# Data parameters
START_DATE = '2026-01-01'

# blueshift.campaign_activity_kpis flag semantics, confirmed with the table owner:
ATTRIBUTION_WINDOW_DAYS = 2  # fix_flag/direct_buy_flag are attributed within 2 days of sent_timestamp
                             # (referral counts specifically get a longer 7-day attribution window)
MATURITY_GATE_DAYS = 45      # the ETL reprocesses a 45-day sliding window after send -- a row's
                             # flag can still be corrected/backfilled until then, after which it's
                             # frozen (until a full table rebuild). A month's flags aren't safe to
                             # trust as final until every send in it clears this window.

# Lifecycle thresholds (days since last checkout), per client_state_detail on curated.checkout_based_client_state_journal
ACTIVE_MAX_DAYS = 120
LAPSED_MAX_DAYS = 365
DORMANT_3YR_MIN_DAYS = 1095  # eligibility cutoff for THIS experiment

# Design parameters
INITIAL_ALPHA = 0.05
N_COMPARISONS = 1  # single comparison: Treatment (Personalization) vs. Control (No Personalization)
ALPHA = INITIAL_ALPHA / N_COMPARISONS  # = 0.05, no Bonferroni needed for a single pairwise A/B
POWER = 0.80
TWO_SIDED = True
N_ARMS = 2
MDE_GRID = [0.03, 0.05, 0.10, 0.15]  # relative lift on reactivation rate
HARM_GRID = [-0.03, -0.05]  # relative drop (Early Stop Condition)

## Step 0 — Confirm lifecycle buckets against the warehouse table

Sanity check `client_state_detail` on `curated.checkout_based_client_state_journal` (current row per client, `is_current = 1`) against the `ACTIVE_MAX_DAYS` / `LAPSED_MAX_DAYS` / `DORMANT_3YR_MIN_DAYS` thresholds, then get the eligible population size for this experiment.

In [2]:
lifecycle_bucket_query = f"""--sql
WITH journal AS (
    SELECT
        client_id,
        client_state_detail,
        date_diff('day', date(last_buyable_checkout_ts), current_date) AS days_since_last_checkout
    FROM curated.checkout_based_client_state_journal
    WHERE is_current = 1
)
SELECT
    CASE
        WHEN client_state_detail = 'Never Active' THEN 'Never Active'
        WHEN days_since_last_checkout <= {ACTIVE_MAX_DAYS} THEN 'Active'
        WHEN days_since_last_checkout <= {LAPSED_MAX_DAYS} THEN 'Lapsed'
        WHEN days_since_last_checkout <= {DORMANT_3YR_MIN_DAYS} THEN 'Dormant'
        ELSE 'Dormant 3+ yrs'
    END AS lifecycle_bucket,
    COUNT(*) AS n_clients
FROM journal
GROUP BY 1
ORDER BY n_clients DESC
"""

lifecycle_bucket_df = query(lifecycle_bucket_query)
lifecycle_bucket_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,lifecycle_bucket,n_clients
0,Never Active,36863824
1,Dormant 3+ yrs,9805722
2,Dormant,2102284
3,Active,1464027
4,Lapsed,868517


The eligible pool for this experiment (**Dormant 3+ yrs, ~9.8M clients**) is comfortably large; population size is not the constraint here.

## Step 1 — Reactivation baseline & eligible daily volume (from `blueshift.campaign_activity_kpis`)

Primary metric = **Reactivation** := `fix_flag = 1 OR direct_buy_flag = 1` on a send. `blueshift.campaign_activity_summary` has no conversion columns at all, so it can't answer this — `kpis` is the enriched version of that table with per-send outcome flags added.

Two things this join needs to get right:
1. **Point-in-time eligibility**: a client's Dormant-3yr status must be evaluated **as of the send**, not as of today — join `campaign_activity_kpis.sent_timestamp` against the journal's `start_timestamp`/`end_timestamp` validity window, not the `is_current` snapshot used in Step 0.
2. **Per-client, not per-send, rate**: a client can receive multiple sends in a month; the metric that matters for sizing is whether *the client* reactivated, not how many of their individual sends carried the flag. Rows are collapsed to `client_id x month` with `MAX(reactivated)` before computing the rate.

`holdout_group = 0` excludes suppressed/holdout sends, since those aren't part of the addressable campaign population.

In [3]:
reactivation_query = f"""--sql
WITH sends AS (
    SELECT
        k.client_id,
        k.sent_timestamp,
        CASE WHEN k.fix_flag = 1 OR k.direct_buy_flag = 1 THEN 1 ELSE 0 END AS reactivated
    FROM blueshift.campaign_activity_kpis k
    WHERE k.execution_date >= DATE '{START_DATE}'
      AND k.holdout_group = 0
),
dormant_3yr_sends AS (
    -- point-in-time eligibility: Dormant 3+ yrs status AS OF the send, via the journal's validity window
    SELECT
        s.client_id,
        s.sent_timestamp,
        s.reactivated
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
),
client_month AS (
    -- collapse to client x month: did THIS client reactivate at all that month
    SELECT
        client_id,
        DATE_TRUNC('month', sent_timestamp) AS month,
        MAX(reactivated) AS client_reactivated
    FROM dormant_3yr_sends
    GROUP BY 1, 2
),
month_days AS (
    SELECT DATE_TRUNC('month', sent_timestamp) AS month, COUNT(DISTINCT CAST(sent_timestamp AS DATE)) AS days_observed
    FROM dormant_3yr_sends
    GROUP BY 1
)
SELECT
    cm.month,
    md.days_observed,
    COUNT(DISTINCT cm.client_id) AS unique_clients_sent,
    SUM(cm.client_reactivated) AS reactivated_clients,
    CAST(SUM(cm.client_reactivated) AS DOUBLE) / COUNT(DISTINCT cm.client_id) AS client_reactivation_rate,
    ROUND(COUNT(DISTINCT cm.client_id) * 1.0 / md.days_observed, 1) AS unique_clients_per_day
FROM client_month cm
JOIN month_days md ON cm.month = md.month
GROUP BY cm.month, md.days_observed
ORDER BY cm.month DESC
"""

reactivation_df = query(reactivation_query)
reactivation_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,unique_clients_sent,reactivated_clients,client_reactivation_rate,unique_clients_per_day
0,2026-07-01 00:00:00.000,15,1146683,4930,0.004299,76445.5
1,2026-06-01 00:00:00.000,30,1204986,9872,0.008193,40166.2
2,2026-05-01 00:00:00.000,31,1243651,12578,0.010114,40117.8
3,2026-04-01 00:00:00.000,30,1198458,14235,0.011878,39948.6
4,2026-03-01 00:00:00.000,31,1196460,16721,0.013975,38595.5
5,2026-02-01 00:00:00.000,28,1188131,13364,0.011248,42433.3
6,2026-01-01 00:00:00.000,31,1338048,15074,0.011266,43162.8


### Picking the reference month — flag finality, not user conversion time

**Observed monthly rates climb the further back you go.** It's tempting to read this as "users take a long time to convert," but the actual `fix_flag`/`direct_buy_flag` attribution window is a fixed **`ATTRIBUTION_WINDOW_DAYS` (2 days)** post-send — conversions past that were never going to be counted regardless of how long we wait.

What actually moves is **flag finality**: per the table owner, the ETL reprocesses a **`MATURITY_GATE_DAYS` (45-day)** sliding window after send, correcting/backfilling flags as more data lands. A month's numbers are only guaranteed final once every send in it has cleared that window; before that, the observed rate can still tick up.

**Rule:** only trust a month once *every* send in it has cleared the gate, i.e. `last_day_of_month + MATURITY_GATE_DAYS <= today`.

In [4]:
from datetime import date, timedelta
from calendar import monthrange

def is_mature(month_ts, maturity_gate_days=MATURITY_GATE_DAYS, today=None):
    today = today or date.today()
    y, m = month_ts.year, month_ts.month
    last_day = date(y, m, monthrange(y, m)[1])
    return last_day + timedelta(days=maturity_gate_days) <= today

reactivation_df['month_ts'] = pd.to_datetime(reactivation_df['month'])
reactivation_df['mature'] = reactivation_df['month_ts'].apply(is_mature)

mature_months = reactivation_df[reactivation_df['mature']].sort_values('month_ts', ascending=False)
REFERENCE_MONTH = mature_months.iloc[0]

BASELINE_RATE = float(REFERENCE_MONTH['client_reactivation_rate'])
DAILY_ELIGIBLE_CLIENTS = float(REFERENCE_MONTH['unique_clients_per_day'])

print(f"REFERENCE_MONTH = {REFERENCE_MONTH['month']}  (most recent month with ETL-final flags, {MATURITY_GATE_DAYS}-day maturity gate)")
print(f"BASELINE_RATE = {BASELINE_RATE:.4%}  |  DAILY_ELIGIBLE_CLIENTS = {DAILY_ELIGIBLE_CLIENTS:,.0f}")
reactivation_df

REFERENCE_MONTH = 2026-05-01 00:00:00.000  (most recent month with ETL-final flags, 45-day maturity gate)
BASELINE_RATE = 1.0114%  |  DAILY_ELIGIBLE_CLIENTS = 40,118


,month,days_observed,unique_clients_sent,reactivated_clients,client_reactivation_rate,unique_clients_per_day,month_ts,mature
0,2026-07-01 00:00:00.000,15,1146683,4930,0.004299,76445.5,2026-07-01,False
1,2026-06-01 00:00:00.000,30,1204986,9872,0.008193,40166.2,2026-06-01,False
2,2026-05-01 00:00:00.000,31,1243651,12578,0.010114,40117.8,2026-05-01,True
3,2026-04-01 00:00:00.000,30,1198458,14235,0.011878,39948.6,2026-04-01,True
4,2026-03-01 00:00:00.000,31,1196460,16721,0.013975,38595.5,2026-03-01,True
5,2026-02-01 00:00:00.000,28,1188131,13364,0.011248,42433.3,2026-02-01,True
6,2026-01-01 00:00:00.000,31,1338048,15074,0.011266,43162.8,2026-01-01,True


## Step 1c — Why `MATURITY_GATE_DAYS = 45`

No DDL, dbt model, or ETL job for `blueshift.campaign_activity_kpis` exists in this codebase. The values below are confirmed against the actual pipeline: `kpi_join.py` (the enrichment job, owned by data engineering) and the `de_blueshift` Futura DAG that schedules it.

- **Attribution windows**: `fix_flag`/`direct_buy_flag`/`style_pass_flag`/`cancelled_autoship_flag`/`kids_fix_flag` are attributed within `ATTRIBUTION_WINDOW_DAYS` (2 days) of `sent_timestamp` — hardcoded in `kpi_join.py`'s join logic. Referral counts (`mens_referrals_count`/`womens_referrals_count`, not used by this experiment's metric) get a longer 7-day window.
- **Flag mutability**: the KPI enrichment task reprocesses a sliding window, rebuilding any row with `execution_key` inside that window and leaving everything older frozen. The `de_blueshift` DAG sets this window to **45 days** for the KPI enrichment step specifically (a separate, shorter 7-day window elsewhere in the same DAG applies only to raw event ingestion, not this flag).

Net effect: a month's `client_reactivation_rate` isn't safe to treat as final until every send in it has cleared the 45-day window — which is exactly what `is_mature()` above checks for.

## Step 2 — Sample size & duration

`n_total_statsmodels` is pairwise, which is exactly what a 2-arm A/B needs. `n_treatment` **is** the per-arm requirement (50/50 split), `total = 2 x n_per_arm`, and each arm accrues `DAILY_ELIGIBLE_CLIENTS / 2` per day.

In [5]:
def size_table(rel_grid, baseline, daily, label):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[0.5],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_2arm'] = df['n_per_arm'] * N_ARMS
    if daily:
        df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
        df['weeks_required'] = (df['days_required'] / 7).round(1)
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_2arm','days_required','weeks_required']
    else:
        cols = ['rel_effect','p_treatment','n_per_arm','n_total_2arm']

    sided = 'two-sided' if TWO_SIDED else 'one-sided'
    print(f"--- {label} (baseline={baseline:.2%}, alpha={ALPHA} (no multiple-comparison correction), power={POWER:.0%}, {sided}) ---")

    return df[cols]

size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE_CLIENTS, 'Positive MDE')

--- Positive MDE (baseline=1.01%, alpha=0.05 (no multiple-comparison correction), power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,+3%,0.010417,1732470,3464940,87,12.4
1,+5%,0.010619,629769,1259538,32,4.6
2,+10%,0.011125,161241,322482,9,1.3
3,+15%,0.011631,73350,146700,4,0.6


In [6]:
# Harm side (Early Stop Condition). With a two-sided test these n's mirror the positive grid;
# shown explicitly to document the harm magnitude the test is powered to detect.
size_table(HARM_GRID, BASELINE_RATE, DAILY_ELIGIBLE_CLIENTS, 'Harm / guardrail')

--- Harm / guardrail (baseline=1.01%, alpha=0.05 (no multiple-comparison correction), power=80%, two-sided) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,-3%,0.00981,1681779,3363558,84,12.0
1,-5%,0.009608,599355,1198710,30,4.3


## Step 3 — Summary for Experiment Design doc

The MDE should be defined in advance, then pasted to document the Power Analysis section in the Experiment Design doc.

In [7]:
TARGET_REL_MDE = 0.10  # NOTE: placeholder relative MDE (10% relative lift on reactivation rate) pending Product sign-off

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[0.5],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED
    )

n_per_arm = int(list(res.values())[0]['n_treatment'])

duration = f"{int(np.ceil(n_per_arm / (DAILY_ELIGIBLE_CLIENTS/N_ARMS)))} days" if DAILY_ELIGIBLE_CLIENTS else 'TODO (run Step 1)'

summary = {
    'Metric Used': f"Reactivation Rate (Fix OR Direct Buy within {ATTRIBUTION_WINDOW_DAYS} days of send)",
    'Population': 'Dormant 3+ yrs clients (no checkout in >1095 days), campaign-eligible',
    'Baseline Value': f"{BASELINE_RATE:.2%} client reactivation rate ({REFERENCE_MONTH['month']}, ETL-final per table-owner-confirmed {MATURITY_GATE_DAYS}-day maturity gate)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.4f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.4f})",
    'One/Two-Sided Test': 'Two-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction needed)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control: No Personalization / Treatment: Personalization)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': duration,
    'Early Stop / harm threshold': f"-{abs(TARGET_REL_MDE):.0%} relative (covered by two-sided sizing)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Reactivation Rate (Fix OR Direct Buy within 2 ...
Population,Dormant 3+ yrs clients (no checkout in >1095 d...
Baseline Value,1.01% client reactivation rate (2026-05-01 00:...
Minimum Detectable Effect,+10% relative (0.0101 -> 0.0111)
One/Two-Sided Test,Two-sided
Significance Level,"0.05 (single comparison, no multiple-compariso..."
Statistical Power,80%
Variant Split %,50% / 50% (Control: No Personalization / Treat...
Minimum Samples by Variant,"161,241"
Minimum Samples total,"322,482"
